# 1. Bronze Layer Ingestion — Detailed Line-by-Line Breakdown

This notebook demonstrates how raw data files stored in Databricks Unity Catalog Volumes are ingested into Delta Lake Bronze tables using Spark SQL.

--- 
## Cell 1: Previewing Raw CSV Data with `read_files`

In [ ]:
%sql
-- Peek at the raw orders file before loading anything
SELECT * FROM read_files(
  '/Volumes/shopstream/core/raw/orders_2026_h1.csv',
  format => 'csv',
  header => true
)
LIMIT 10

### Line-by-Line Code Explanation:

1. **`%sql`**
   - **Purpose:** A Databricks magic command that switches the cell execution language from the notebook's default (Python) to Spark SQL.

2. **`-- Peek at the raw orders file before loading anything`**
   - **Purpose:** SQL single-line comment describing the intent of the statement.

3. **`SELECT * FROM read_files(`**
   - **`SELECT *`**: Projects all columns discovered inside the source file.
   - **`read_files()`**: A built-in Spark SQL table-valued function designed to read raw files directly from Cloud Storage or Unity Catalog Volumes without creating an external table beforehand.

4. **`'/Volumes/shopstream/core/raw/orders_2026_h1.csv',`**
   - **Purpose:** Specifies the absolute Unity Catalog Volume path pointing to the raw input CSV file (`catalog = shopstream`, `schema = core`, `volume = raw`).

5. **`format => 'csv',`**
   - **Purpose:** Named argument defining the file parser format. Here, it explicitly instructs Databricks to use the CSV file reader.

6. **`header => true`**
   - **Purpose:** Configures the CSV parser to use the first row of the input file as header names (`order_line_id`, `order_id`, `customer_id`, etc.) rather than generic column names like `_c0`, `_c1`.

7. **`)`**
   - **Purpose:** Closes the parameters list for the `read_files()` function.

8. **`LIMIT 10`**
   - **Purpose:** Restricts the returned dataset to the top 10 rows. This prevents unnecessary processing or large data transfers during exploratory data checks.

--- 
## Cell 2: Creating and Ingesting the Bronze Orders Table (`COPY INTO`)

In [ ]:
%sql
-- Bronze orders: create once, then load with COPY INTO (idempotent)
CREATE TABLE IF NOT EXISTS shopstream.core.bronze_orders;

COPY INTO shopstream.core.bronze_orders
FROM '/Volumes/shopstream/core/raw/orders_2026_h1.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

### Line-by-Line Code Explanation:

1. **`CREATE TABLE IF NOT EXISTS shopstream.core.bronze_orders;`**
   - **`CREATE TABLE IF NOT EXISTS`**: Ensures the SQL statement executes without failing if the table already exists in the catalog.
   - **`shopstream.core.bronze_orders`**: Fully qualified 3-level namespace (`catalog.schema.table`) where the Delta table will be registered in Unity Catalog.
   - *Note:* Since no schema definition is provided, an empty Delta table shell is instantiated until populated by the `COPY INTO` infer process.

2. **`COPY INTO shopstream.core.bronze_orders`**
   - **`COPY INTO`**: An idempotent Spark SQL command used to load data continuously or incrementally from a file path into a Delta Lake table. It tracks processed files to avoid duplicate rows upon re-execution.

3. **`FROM '/Volumes/shopstream/core/raw/orders_2026_h1.csv'`**
   - **Purpose:** Designates the source path containing raw input data.

4. **`FILEFORMAT = CSV`**
   - **Purpose:** Specifies that the source files are in CSV format.

5. **`FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')`**
   - **`'header' = 'true'`**: Reads column names directly from the first line of the CSV.
   - **`'inferSchema' = 'true'`**: Automatically detects data types (e.g., parsing integers, timestamps, doubles) rather than casting all fields as strings.
   - **`'mergeSchema' = 'true'`**: Allows schema evolution so new columns present in incoming CSV files automatically update the target Delta table schema.

6. **`COPY_OPTIONS ('mergeSchema' = 'true')`**
   - **Purpose:** Instructs the `COPY INTO` engine explicitly to evolve the target Delta table's schema definition if new attributes appear during ingestion.

--- 
## Cell 3: Ingesting Dimension Data (`Customers` and `Products`)

In [ ]:
%sql
-- Same pattern for the two dimension files
CREATE TABLE IF NOT EXISTS shopstream.core.bronze_customers;

COPY INTO shopstream.core.bronze_customers
FROM '/Volumes/shopstream/core/raw/customers.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

CREATE TABLE IF NOT EXISTS shopstream.core.bronze_products;

COPY INTO shopstream.core.bronze_products
FROM '/Volumes/shopstream/core/raw/products.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

### Line-by-Line Code Explanation:

1. **`CREATE TABLE IF NOT EXISTS shopstream.core.bronze_customers;`**
   - Creates the target target Delta table shell for raw customer dimension records.

2. **`COPY INTO shopstream.core.bronze_customers FROM '/Volumes/.../customers.csv' ...`**
   - Ingests customer data into Delta format with header parsing, schema inference, and automatic schema evolution enabled.

3. **`CREATE TABLE IF NOT EXISTS shopstream.core.bronze_products;`**
   - Creates the target Delta table shell for raw product dimension records.

4. **`COPY INTO shopstream.core.bronze_products FROM '/Volumes/.../products.csv' ...`**
   - Ingests product catalog data into the corresponding Delta table using idempotent loading options.

--- 
## Cell 4: Verification & Audit Row Counts

In [ ]:
%sql
-- The bronze layer is in. Three tables, raw shapes preserved.
SELECT 'bronze_orders' AS table_name, COUNT(*) AS row_count FROM shopstream.core.bronze_orders
UNION ALL
SELECT 'bronze_customers', COUNT(*) FROM shopstream.core.bronze_customers
UNION ALL
SELECT 'bronze_products', COUNT(*) FROM shopstream.core.bronze_products

### Line-by-Line Code Explanation:

1. **`SELECT 'bronze_orders' AS table_name, COUNT(*) AS row_count FROM shopstream.core.bronze_orders`**
   - Assigns the string literal `'bronze_orders'` to column `table_name` and calculates the total row count of `shopstream.core.bronze_orders` as `row_count`.

2. **`UNION ALL`**
   - Combines result sets vertically across queries without evaluating or removing duplicate rows, maximizing query speed.

3. **`SELECT 'bronze_customers', COUNT(*) FROM shopstream.core.bronze_customers`**
   - Appends the row count of the `bronze_customers` table.

4. **`SELECT 'bronze_products', COUNT(*) FROM shopstream.core.bronze_products`**
   - Appends the row count of the `bronze_products` table, outputting a consolidated 3-row audit report.